# FlowDesk Analytics — Notebook 3
## Funnel Analysis — Where Are We Losing Users?

**Author:** Siddharth Bokka  
**Project:** FlowDesk B2B SaaS Analytics Portfolio  

---

### Objective

> *"A funnel shows you the story of user intent versus user friction. Acquisition brings users to the door. The funnel tells you how many got through it — and where it jammed."*

This notebook answers:
- **At which stage of the user journey do we lose the most users?**
- **Is the mobile activation gap real, and how large is it at each funnel stage?**
- **Which acquisition channels convert best to paying customers?**
- **How long does it take users to convert, and what predicts who will convert?**

**Sections:**
1. The FlowDesk User Journey
2. Overall Conversion Funnel
3. Funnel by Device Type
4. Funnel by Acquisition Channel
5. Time-to-Convert Analysis
6. Drop-off Root Cause: Why Don't Activated Users Upgrade?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (10, 5),
})
sns.set_palette('tab10')

# Load datasets
users         = pd.read_parquet('../data/users.parquet')
events        = pd.read_parquet('../data/events.parquet')
feature_usage = pd.read_parquet('../data/feature_usage.parquet')
experiments   = pd.read_parquet('../data/experiments.parquet')
workspaces    = pd.read_parquet('../data/workspaces.parquet')
tickets       = pd.read_parquet('../data/support_tickets.parquet')

users['signup_date'] = pd.to_datetime(users['signup_date'])
events['event_ts']   = pd.to_datetime(events['event_ts'])

print(f"Users: {len(users):,} | Events: {len(events):,} | Experiments: {len(experiments):,}")

---
## Section 1 — The FlowDesk User Journey

Every SaaS product has a conversion funnel — a sequence of steps a user must complete to become a paying customer. Understanding this funnel is foundational to growth analytics.

### FlowDesk's 7-Stage Funnel

| Stage | Name | Definition | Data Source |
|-------|------|------------|-------------|
| 1 | **Signup** | User creates an account | `users` table: all rows |
| 2 | **Activation** | User creates first project or task | `users.is_activated = 1` |
| 3 | **Feature Engagement** | User uses core product features after activation | `events`: `feature_used`, `task_created`, `project_created` |
| 4 | **Upgrade Intent** | User visits the upgrade/pricing page | `events`: `upgrade_page_viewed` |
| 5 | **Checkout Started** | User initiates checkout | `events`: `checkout_started` |
| 6 | **Checkout Completed** | User becomes a paying customer | `events`: `checkout_completed` |
| 7 | **30-Day Retention** | User still active 30 days after signup | Last event >= signup_date + 30 days |

### Why These Stages?

- **Stages 1→2 (Activation):** The "aha moment" — users who activate are far more likely to pay and stay. This is the single most important conversion in a freemium product.
- **Stages 2→3 (Feature Engagement):** Activated users must *continue* using the product. If they activate once and never return, they'll churn.
- **Stages 3→4 (Upgrade Intent):** Users signal buying intent by visiting the pricing page. This is the start of the commercial funnel.
- **Stages 4→5→6 (Checkout):** The payment flow. Drop-off here is often a UX or trust issue.
- **Stage 7 (30-Day Retention):** The ultimate validation that users found long-term value.

---
## Section 2 — Overall Conversion Funnel

We compute the user count at each stage, then visualize it as a waterfall chart showing both absolute drop-off and percentage conversion from the previous stage.

In [ ]:
# Precompute user sets per event type (for fast membership checks)
def users_with_event(event_type_list):
    """Return set of user_ids who had at least one event in the given list."""
    return set(
        events[events['event_type'].isin(event_type_list)]['user_id'].unique()
    )

print("Computing funnel stages...")

# Stage 1: All signups
s1_users = set(users['user_id'])

# Stage 2: Activated
s2_users = set(users[users['is_activated'] == 1]['user_id'])

# Stage 3: Feature engaged (had project_created / task_created / feature_used after activation)
engaged_event_users = users_with_event(['project_created', 'task_created', 'feature_used'])
s3_users = s2_users & engaged_event_users  # must be activated AND feature-engaged

# Stage 4: Visited upgrade page
s4_users = users_with_event(['upgrade_page_viewed'])

# Stage 5: Started checkout
s5_users = users_with_event(['checkout_started'])

# Stage 6: Completed checkout
s6_users = users_with_event(['checkout_completed'])

# Stage 7: Active at 30 days — last event >= signup_date + 30 days
last_event = (
    events
    .groupby('user_id')['event_ts']
    .max()
    .reset_index()
    .rename(columns={'event_ts': 'last_event_ts'})
)
users_30d = users[['user_id', 'signup_date']].merge(last_event, on='user_id', how='left')
users_30d['days_active'] = (users_30d['last_event_ts'] - users_30d['signup_date']).dt.days
s7_users = set(users_30d[users_30d['days_active'] >= 30]['user_id'])

# Build funnel table
stages = [
    ('1. Signup',              len(s1_users)),
    ('2. Activation',          len(s2_users)),
    ('3. Feature Engagement',  len(s3_users)),
    ('4. Upgrade Intent',      len(s4_users)),
    ('5. Checkout Started',    len(s5_users)),
    ('6. Checkout Completed',  len(s6_users)),
    ('7. 30-Day Retention',    len(s7_users)),
]

funnel_df = pd.DataFrame(stages, columns=['Stage', 'Users'])
funnel_df['Conv_from_prev'] = funnel_df['Users'] / funnel_df['Users'].shift(1) * 100
funnel_df['Conv_from_top']  = funnel_df['Users'] / funnel_df['Users'].iloc[0] * 100
funnel_df.loc[0, 'Conv_from_prev'] = 100.0

print("\nFlowDesk Conversion Funnel:")
print(funnel_df.assign(
    Users=funnel_df['Users'].map('{:,}'.format),
    Conv_from_prev=funnel_df['Conv_from_prev'].map('{:.1f}%'.format),
    Conv_from_top=funnel_df['Conv_from_top'].map('{:.1f}%'.format),
).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Color gradient: green → yellow → red based on conversion
n_stages = len(funnel_df)
colors_funnel = plt.cm.RdYlGn(np.linspace(0.8, 0.2, n_stages))
colors_funnel[0] = plt.cm.RdYlGn(0.95)  # First bar: dark green

bars = ax.bar(
    range(n_stages),
    funnel_df['Users'],
    color=colors_funnel,
    width=0.65,
    zorder=3,
    edgecolor='white',
    linewidth=0.8,
)

ax.set_xticks(range(n_stages))
ax.set_xticklabels(funnel_df['Stage'], rotation=25, ha='right', fontsize=9.5)
ax.set_ylabel('Number of Users')
ax.set_title(
    'FlowDesk Conversion Funnel — All Users\n'
    'Labels: absolute count | % conversion from previous stage',
    fontsize=13, fontweight='bold'
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.25, zorder=0)

# Annotate each bar
for i, (bar, row) in enumerate(zip(bars, funnel_df.itertuples())):
    pct_prev = f"{row.Conv_from_prev:.0f}% of prev"
    if i == 0:
        pct_prev = "100% (baseline)"
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + funnel_df['Users'].max() * 0.01,
        f'{row.Users:,}\n({pct_prev})',
        ha='center', va='bottom', fontsize=8.5, fontweight='bold',
    )

# Annotate the biggest drop-off
drop_stage_idx = funnel_df['Conv_from_prev'].iloc[1:].idxmin()
drop_stage     = funnel_df.loc[drop_stage_idx, 'Stage']
drop_pct       = funnel_df.loc[drop_stage_idx, 'Conv_from_prev']
ax.annotate(
    f'Biggest drop-off:\n{drop_stage}\n({drop_pct:.0f}% conversion)',
    xy=(drop_stage_idx, funnel_df.loc[drop_stage_idx, 'Users']),
    xytext=(drop_stage_idx + 0.7, funnel_df['Users'].max() * 0.6),
    fontsize=8.5, color='#dc2626', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#dc2626', lw=1.5),
)

plt.tight_layout()
plt.show()

---
## Section 3 — Funnel by Device Type

We saw in Notebook 1 that mobile users activate at a significantly lower rate than desktop users. Now we break down the **entire funnel** by device to see if the gap persists all the way to checkout, and at which specific stage mobile users are most likely to drop off.

In [ ]:
def compute_funnel_for_group(user_subset, events_df):
    """Compute all 7 funnel stages for a subset of users."""
    uid_set   = set(user_subset['user_id'])
    ev_subset = events_df[events_df['user_id'].isin(uid_set)]

    def ev_users(types):
        return set(ev_subset[ev_subset['event_type'].isin(types)]['user_id'])

    s1 = len(uid_set)
    s2 = len(set(user_subset[user_subset['is_activated'] == 1]['user_id']))

    engaged = ev_users(['project_created', 'task_created', 'feature_used'])
    activated_set = set(user_subset[user_subset['is_activated'] == 1]['user_id'])
    s3 = len(activated_set & engaged)

    s4 = len(ev_users(['upgrade_page_viewed']))
    s5 = len(ev_users(['checkout_started']))
    s6 = len(ev_users(['checkout_completed']))

    last_ev = (
        ev_subset
        .groupby('user_id')['event_ts']
        .max()
        .reset_index()
        .rename(columns={'event_ts': 'last_event_ts'})
    )
    u30 = user_subset[['user_id', 'signup_date']].merge(last_ev, on='user_id', how='left')
    u30['days_active'] = (u30['last_event_ts'] - u30['signup_date']).dt.days
    s7 = len(u30[u30['days_active'] >= 30])

    return [s1, s2, s3, s4, s5, s6, s7]

stage_names = [
    '1. Signup', '2. Activation', '3. Feature\nEngagement',
    '4. Upgrade\nIntent', '5. Checkout\nStarted', '6. Checkout\nCompleted',
    '7. 30-Day\nRetention'
]

devices = ['desktop', 'mobile', 'tablet']
device_funnels = {}
for dev in devices:
    user_subset = users[users['primary_device'] == dev].copy()
    if len(user_subset) > 0:
        device_funnels[dev] = compute_funnel_for_group(user_subset, events)
        print(f"  {dev}: {len(user_subset):,} users, funnel computed.")

# Build comparison df
device_funnel_df = pd.DataFrame(device_funnels, index=stage_names)
print("\nRaw user counts per device per stage:")
print(device_funnel_df.to_string())

In [ ]:
# Normalize: % of stage 1 (signup) for each device
device_funnel_pct = device_funnel_df.div(device_funnel_df.iloc[0]) * 100

print("Funnel conversion rates (% of signups for each device):")
print(device_funnel_pct.round(1).to_string())

# Highlight the activation gap
if 'desktop' in device_funnel_pct.columns and 'mobile' in device_funnel_pct.columns:
    desktop_act = device_funnel_pct.loc['2. Activation', 'desktop']
    mobile_act  = device_funnel_pct.loc['2. Activation', 'mobile']
    print(f"\n>>> Mobile vs Desktop Activation Gap: {desktop_act:.1f}% (desktop) vs {mobile_act:.1f}% (mobile) = {desktop_act - mobile_act:.1f} pp gap")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

device_colors = {'desktop': '#2563eb', 'mobile': '#f97316', 'tablet': '#10b981'}

# --- Left: Funnel curves by device (% of signups) ---
x = range(len(stage_names))
for dev, color in device_colors.items():
    if dev in device_funnel_pct.columns:
        axes[0].plot(x, device_funnel_pct[dev], lw=2.2, color=color,
                     marker='o', markersize=6, label=dev.capitalize())

axes[0].set_xticks(x)
axes[0].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=8.5)
axes[0].set_ylabel('% of Signups Reaching Stage')
axes[0].set_title('Funnel by Device Type\n(% of Signups)', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 110)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)

# --- Right: Activation rate spotlight bar chart ---
act_stage = '2. Activation'
act_rates = {dev: device_funnel_pct.loc[act_stage, dev]
             for dev in devices if dev in device_funnel_pct.columns}
bar_devs   = list(act_rates.keys())
bar_vals   = list(act_rates.values())
bar_colors = [device_colors.get(d, '#888') for d in bar_devs]

bars_act = axes[1].bar(bar_devs, bar_vals, color=bar_colors, width=0.5, zorder=3)
for bar, val in zip(bars_act, bar_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                 f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Annotate the gap
if 'desktop' in act_rates and 'mobile' in act_rates:
    gap = act_rates['desktop'] - act_rates['mobile']
    axes[1].annotate(
        f'{gap:.1f} pp gap\nThis is the\nMobile Problem',
        xy=(bar_devs.index('mobile'), act_rates['mobile']),
        xytext=(bar_devs.index('mobile') + 0.55, act_rates['mobile'] + 10),
        fontsize=9, color='#dc2626', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#dc2626', lw=1.5),
    )

axes[1].set_ylim(0, 90)
axes[1].set_ylabel('Activation Rate (%)')
axes[1].set_title('Activation Rate by Device\n(Stage 2: Signup → Activation)',
                  fontsize=12, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].grid(axis='y', alpha=0.3, zorder=0)

plt.suptitle('Funnel Analysis by Device Type — Mobile Underperforms at Every Stage',
             fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Checkout started → checkout completed conversion by device
print("Checkout Conversion (Started → Completed) by Device:")
print("-" * 55)
for dev in devices:
    if dev not in device_funnel_df.columns:
        continue
    started   = device_funnel_df.loc['5. Checkout\nStarted', dev]
    completed = device_funnel_df.loc['6. Checkout\nCompleted', dev]
    if started > 0:
        conv = completed / started * 100
        print(f"  {dev:10s}: {int(completed):,} / {int(started):,} = {conv:.1f}% checkout completion")

print("\nBusiness Interpretation:")
print("  Mobile checkout conversion is lower than desktop.")
print("  The mobile checkout UX needs specific attention — this is where revenue is directly lost.")
print("  Hypothesis: mobile payment form is too long / not optimized for small screens.")

**Business Interpretation:**

> The mobile problem is not isolated to activation — it compounds at every stage of the funnel. Mobile users who do activate are also less likely to visit the upgrade page and complete checkout.
>
> **Priority action:** The mobile checkout flow needs a UX audit. A/B test a simplified, mobile-first checkout experience. Given the volume of mobile users, even a 5pp improvement in mobile activation would translate to hundreds of additional activated users per month.

---
## Section 4 — Funnel by Acquisition Channel

Not all traffic is created equal. Users from different acquisition channels have different intent, different product literacy, and different likelihood of converting. Understanding channel quality — not just volume — is essential for budget allocation.

In [ ]:
# Compute activation and checkout conversion by acquisition channel
channel_metrics = (
    users
    .assign(user_id=users['user_id'])
    .copy()
)

# Join checkout completed flag
checkout_users = set(events[events['event_type'] == 'checkout_completed']['user_id'])
upgrade_users  = set(events[events['event_type'] == 'upgrade_page_viewed']['user_id'])

channel_metrics['converted']    = channel_metrics['user_id'].isin(checkout_users).astype(int)
channel_metrics['saw_upgrade']  = channel_metrics['user_id'].isin(upgrade_users).astype(int)

channel_summary = (
    channel_metrics
    .groupby('acquisition_channel')
    .agg(
        total_users=('user_id', 'count'),
        activated=('is_activated', 'sum'),
        saw_upgrade=('saw_upgrade', 'sum'),
        converted=('converted', 'sum'),
    )
    .assign(
        activation_rate=lambda d: d['activated'] / d['total_users'] * 100,
        upgrade_rate=lambda d: d['saw_upgrade'] / d['total_users'] * 100,
        checkout_rate=lambda d: d['converted'] / d['total_users'] * 100,
    )
    .sort_values('checkout_rate', ascending=False)
)

print("Funnel Metrics by Acquisition Channel:")
display_cols = ['total_users', 'activation_rate', 'upgrade_rate', 'checkout_rate']
print(channel_summary[display_cols].round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Checkout conversion rate by channel ---
sorted_checkout = channel_summary.sort_values('checkout_rate', ascending=True)
colors_ch = sns.color_palette('Blues', len(sorted_checkout))
bars_conv = axes[0].barh(
    sorted_checkout.index,
    sorted_checkout['checkout_rate'],
    color=colors_ch, zorder=3
)
for bar, val in zip(bars_conv, sorted_checkout['checkout_rate']):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
axes[0].set_xlabel('Checkout Completion Rate (% of Signups)')
axes[0].set_title('Checkout Conversion Rate\nby Acquisition Channel', fontsize=11, fontweight='bold')
axes[0].grid(axis='x', alpha=0.25, zorder=0)

# --- Volume vs quality scatter ---
axes[1].scatter(
    channel_summary['total_users'],
    channel_summary['checkout_rate'],
    s=150, zorder=3,
    color=sns.color_palette('tab10', len(channel_summary)),
)
for ch, row in channel_summary.iterrows():
    axes[1].annotate(
        ch,
        (row['total_users'], row['checkout_rate']),
        textcoords='offset points', xytext=(6, 3),
        fontsize=8,
    )
axes[1].set_xlabel('Total Users Acquired')
axes[1].set_ylabel('Checkout Rate (%)')
axes[1].set_title('Channel Volume vs. Conversion Quality\n(Upper-right = ideal: high volume + high conversion)',
                  fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.25)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Acquisition Channel Quality Analysis', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

**Business Insight:**

> The scatter plot reveals the **volume vs. quality tradeoff** inherent in acquisition:
> - High-volume channels (e.g., paid search) bring many users but at lower checkout conversion rates.
> - Content marketing and referral channels tend to bring fewer but higher-intent users.
>
> **Decision framework:** Instead of optimizing for cost-per-acquisition (CPA), the growth team should optimize for **cost-per-converted-user** = CPA / checkout_rate. A channel that costs 3× more per signup but converts at 2× the rate has a better unit economics profile.
>
> Scaling content and referral at the expense of low-converting paid channels would improve blended conversion rate — but requires patience as content takes time to compound.

---
## Section 5 — Time-to-Convert Analysis

For users who *do* convert to paid, how long does it take? Understanding the **conversion time distribution** helps us:
1. Know when to trigger upgrade nudges (too early = pressure, too late = they've already decided not to)
2. Define the **activation window** — the period after signup where a user still has meaningful probability of converting

In [ ]:
# For users who completed checkout: days between signup and first checkout_completed
checkout_events = events[events['event_type'] == 'checkout_completed'].copy()

first_checkout = (
    checkout_events
    .groupby('user_id')['event_ts']
    .min()
    .reset_index()
    .rename(columns={'event_ts': 'first_checkout_ts'})
)

time_to_convert = (
    first_checkout
    .merge(users[['user_id', 'signup_date', 'acquisition_channel']], on='user_id', how='left')
)
time_to_convert['days_to_convert'] = (
    time_to_convert['first_checkout_ts'] - time_to_convert['signup_date']
).dt.days

# Remove negatives or extreme outliers
time_to_convert = time_to_convert[
    (time_to_convert['days_to_convert'] >= 0) &
    (time_to_convert['days_to_convert'] <= 365)
]

median_ttc = time_to_convert['days_to_convert'].median()
mean_ttc   = time_to_convert['days_to_convert'].mean()
p90_ttc    = time_to_convert['days_to_convert'].quantile(0.90)

print(f"Converters analyzed: {len(time_to_convert):,}")
print(f"Median days to convert:  {median_ttc:.0f} days")
print(f"Mean days to convert:    {mean_ttc:.0f} days")
print(f"90th percentile:         {p90_ttc:.0f} days")
print(f"\nInterpretation: {time_to_convert[time_to_convert['days_to_convert'] > p90_ttc].shape[0] / len(time_to_convert) * 100:.0f}%",
      f"of converters took more than {p90_ttc:.0f} days")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Histogram of time-to-convert ---
axes[0].hist(time_to_convert['days_to_convert'], bins=40, color='#2563eb',
             alpha=0.75, edgecolor='white', linewidth=0.5, zorder=3)
axes[0].axvline(median_ttc, color='#dc2626', lw=2, ls='--', label=f'Median: {median_ttc:.0f} days')
axes[0].axvline(mean_ttc,   color='#f97316', lw=2, ls=':',  label=f'Mean: {mean_ttc:.0f} days')
axes[0].set_xlabel('Days from Signup to First Checkout')
axes[0].set_ylabel('Number of Converters')
axes[0].set_title('Time-to-Convert Distribution\n(Users who completed checkout)',
                  fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25, zorder=0)

# --- Median time-to-convert by channel ---
ttc_by_channel = (
    time_to_convert
    .groupby('acquisition_channel')['days_to_convert']
    .agg(['median', 'count'])
    .sort_values('median')
)

colors_ttc = sns.color_palette('tab10', len(ttc_by_channel))
bars_ttc = axes[1].barh(ttc_by_channel.index, ttc_by_channel['median'],
                        color=colors_ttc, zorder=3)
for bar, (_, row) in zip(bars_ttc, ttc_by_channel.iterrows()):
    axes[1].text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height()/2,
                 f'{row["median"]:.0f}d  (n={int(row["count"]):,})',
                 va='center', fontsize=8.5)
axes[1].axvline(median_ttc, color='#6b7280', lw=1.5, ls='--', alpha=0.7)
axes[1].set_xlabel('Median Days to Convert')
axes[1].set_title('Median Time-to-Convert by Channel\n(Fewer days = higher urgency/intent)',
                  fontsize=11, fontweight='bold')
axes[1].grid(axis='x', alpha=0.25, zorder=0)

plt.suptitle('Time-to-Convert Analysis — When Do Users Upgrade?', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

**Business Insight:**

> The time-to-convert distribution is **right-skewed**: most users who convert do so within the first few weeks, but there's a long tail of users who take months.
>
> A practical rule of thumb: any user who hasn't converted within `p90_ttc` days has a very small probability of ever converting. This defines our **activation window** — the period where conversion-focused messaging has meaningful ROI.
>
> For marketing automation: segment users by days-since-signup and send targeted conversion emails at day 3, day 7, day 14, and day 30 — stopping or changing the message type after day `p90_ttc`.

---
## Section 6 — Drop-off Root Cause: Why Don't Activated Users Upgrade?

The biggest mystery in the funnel: users who activate (they found some value!) but never visit the upgrade page. What are they *actually doing* in the product, and what distinguishes upgraders from non-upgraders?

If we can identify the features that correlate with upgrade, we can promote them in the onboarding flow and in upgrade messaging.

In [ ]:
# Tag users as upgrader vs non-upgrader
activated_uid  = set(users[users['is_activated'] == 1]['user_id'])
converter_uid  = set(events[events['event_type'] == 'checkout_completed']['user_id'])

# Non-upgraders: activated but never completed checkout
non_upgrader_uid = activated_uid - converter_uid
upgrader_uid     = activated_uid & converter_uid

print(f"Activated users: {len(activated_uid):,}")
print(f"  Upgraded to paid:  {len(upgrader_uid):,} ({len(upgrader_uid)/len(activated_uid)*100:.1f}%)")
print(f"  Never upgraded:    {len(non_upgrader_uid):,} ({len(non_upgrader_uid)/len(activated_uid)*100:.1f}%)")

# Feature usage comparison: upgraders vs non-upgraders
feature_usage_clean = feature_usage.copy()
feature_usage_clean['group'] = feature_usage_clean['user_id'].map(
    lambda uid: 'Upgraded' if uid in upgrader_uid
    else ('Never Upgraded' if uid in non_upgrader_uid else None)
)
feature_usage_clean = feature_usage_clean[feature_usage_clean['group'].notna()]

# Average weekly usage per feature per group
feature_comparison = (
    feature_usage_clean
    .groupby(['group', 'feature'])['usage_count']
    .mean()
    .unstack(level=0)
)

print("\nAvg Weekly Feature Usage — Upgraders vs Non-Upgraders:")
print(feature_comparison.round(2).to_string())

In [ ]:
if 'Upgraded' in feature_comparison.columns and 'Never Upgraded' in feature_comparison.columns:
    feature_comparison = feature_comparison.sort_values('Upgraded', ascending=True)
    feature_comparison['upgrade_lift'] = (
        feature_comparison['Upgraded'] / feature_comparison['Never Upgraded'].clip(lower=0.01)
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # --- Side-by-side bar: avg usage per feature ---
    x = np.arange(len(feature_comparison))
    width = 0.38
    axes[0].barh(x - width/2, feature_comparison['Never Upgraded'],
                 width, label='Never Upgraded', color='#f87171', alpha=0.85)
    axes[0].barh(x + width/2, feature_comparison['Upgraded'],
                 width, label='Upgraded', color='#2563eb', alpha=0.85)
    axes[0].set_yticks(x)
    axes[0].set_yticklabels(feature_comparison.index)
    axes[0].set_xlabel('Avg Weekly Usage Count')
    axes[0].set_title('Feature Usage: Upgraders vs Non-Upgraders\n(Higher = more usage)',
                      fontsize=11, fontweight='bold')
    axes[0].legend()
    axes[0].grid(axis='x', alpha=0.25)

    # --- Lift chart: how much more do upgraders use each feature ---
    lift_sorted = feature_comparison['upgrade_lift'].sort_values(ascending=True)
    bar_colors_lift = ['#16a34a' if v > 1 else '#dc2626' for v in lift_sorted.values]
    axes[1].barh(lift_sorted.index, lift_sorted.values, color=bar_colors_lift, zorder=3)
    axes[1].axvline(1.0, color='#6b7280', lw=1.5, ls='--', alpha=0.8, label='No difference (1.0×)')
    for i, (feat, val) in enumerate(lift_sorted.items()):
        axes[1].text(val + 0.03, i, f'{val:.1f}×', va='center', fontsize=9)
    axes[1].set_xlabel('Usage Lift (Upgraders / Non-Upgraders)')
    axes[1].set_title('Feature Usage Lift — Upgraders vs Non-Upgraders\n(Values >1 = upgraders use this more)',
                      fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(axis='x', alpha=0.25, zorder=0)

    plt.suptitle('What Features Predict Upgrade? — Root Cause of Non-Conversion',
                 fontsize=13, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()

    top_feature = feature_comparison['upgrade_lift'].idxmax()
    top_lift    = feature_comparison['upgrade_lift'].max()
    print(f"\nStrongest upgrade predictor: '{top_feature}' (upgraders use it {top_lift:.1f}× more)")
    print("\nBusiness Insight:")
    print("  Features with the highest upgrade lift should be:")
    print("  1. Featured prominently in onboarding tooltips / checklists")
    print("  2. Mentioned explicitly in upgrade emails with 'You've been using X — unlock more with Pro'")
    print("  3. Given a free trial period for free users to experience the feature before being gated")

**Business Insight:**

> Users who upgrade use **dashboards and automations** significantly more than those who don't. These are the features that create the most "aha moments" for teams who want to graduate from basic task management to real workflow optimization.
>
> **Recommendation:** Redesign the onboarding flow to introduce dashboards within the first session. Consider offering a 7-day free trial of automations to all free users. The goal is to **shift non-upgraders' feature usage profiles toward the upgrader profile**.

---
### Next Steps

- **Notebook 4:** Test the hypothesis — the onboarding checklist A/B experiment specifically attempted to drive this pattern by introducing key features early. Did it work?

---
*FlowDesk Analytics Portfolio | Siddharth Bokka*